# 02 — Eventos unificados e pontos recorrentes (dias 3–4)

Pré-requisitos: `python -m src.pontos geocodificar` e `python -m src.pontos unificar ...`.

In [ ]:
# Preparação: permite importar src/ a partir da pasta notebooks/
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import config

pd.set_option("display.max_columns", 50)
print("Raiz do projeto:", RAIZ)

In [ ]:
eventos = pd.read_parquet(config.ARQ_EVENTOS)
print(len(eventos), "eventos")
print(eventos["fonte"].value_counts())
print(eventos["precisao_geocod"].value_counts())
print(eventos["zona"].value_counts(dropna=False))

## Contagem por ano — cidade × Zona Norte (justificativa do recorte)

In [ ]:
from src.pontos import contar_eventos_por_ano

tabela = contar_eventos_por_ano(eventos)
display(tabela)
tabela.filter(like="cidade_").sum(axis=1).to_frame("cidade").join(
    tabela.filter(like="norte_").sum(axis=1).to_frame("zona_norte")).plot.bar(figsize=(9, 3))

## Sensibilidade do agrupamento
Quantos pontos surgem com raios e mínimos diferentes?

In [ ]:
from src.pontos import identificar_pontos

linhas = []
for raio in (150, 200, 300):
    for minimo in (2, 3, 5):
        c = identificar_pontos(eventos, raio_m=raio, min_eventos=minimo)
        linhas.append({"raio_m": raio, "min_eventos": minimo, "pontos": len(c),
                       "eventos_cobertos": int(c["n_eventos"].sum())})
pd.DataFrame(linhas)

In [ ]:
pontos = pd.read_csv(config.ARQ_PONTOS)
print(len(pontos), "pontos;", (pontos["zona"] == "NORTE").sum(), "na Zona Norte;",
      pontos["celula_id"].nunique(), "células de chuva")
display(pontos.head(20))
pontos.describe()

In [ ]:
# Mapa simples (lat/lon). Para um mapa de fundo: pip install contextily
fig, ax = plt.subplots(figsize=(7, 7))
amostra = eventos.sample(min(len(eventos), 5000), random_state=0)
ax.scatter(amostra["lon"], amostra["lat"], s=2, alpha=0.2, color="gray", label="eventos")
for zona, g in pontos.groupby("zona"):
    ax.scatter(g["lon"], g["lat"], s=10 + g["n_eventos"], label=f"pontos {zona}")
ax.set_aspect("equal")
ax.legend(fontsize=8)
ax.set_title("Pontos recorrentes")

Anote em `DATASET.md` (seção 4): raio, mínimo, número de pontos e distribuição por zona.